In [ ]:
import os
import pandas as pd
import numpy as np

BASE        = '/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
CLEAN_DIR   = f'{BASE}/amia/clean_data'      
MATRIX_DIR  = f'{BASE}/amia/matrix'          
NEWSURV_DIR = f'{BASE}/amia/new_survey'     

os.makedirs(NEWSURV_DIR, exist_ok=True)
print(f"[Dir] {NEWSURV_DIR} ready")

N_TOTAL         = 267747
RESPONSE_THR    = 0.50   
NUMERIC_QID_THR = 0.50

DUP_QUESTIONS = [
    'Gender: Gender Identity',
    'Biological Sex At Birth: Sex At Birth',
    'Black: Black Specific', 'Hispanic: Hispanic Specific', 'AIAN: AIAN Specific',
    'MENA: MENA Specific', 'NHPI: NHPI Specific', 'Asian: Asian Specific',
    'White: White Specific',
]

In [ ]:
def _load_survey(timeframe, TIME='survey_datetime'):
    s_pos = pd.read_csv(f'{CLEAN_DIR}/clean_survey_{timeframe}.csv', low_memory=False)
    s_neg = pd.read_csv(f'{CLEAN_DIR}/clean_negative_survey_{timeframe}.csv', low_memory=False)
    print(f"[Load] pos: {s_pos.shape} | neg: {s_neg.shape}")

    s_pos[TIME] = pd.to_datetime(s_pos[TIME], errors='coerce').dt.normalize()
    s_neg[TIME] = pd.to_datetime(s_neg[TIME], errors='coerce').dt.normalize()

    all_cols = set(s_pos.columns) | set(s_neg.columns)
    for c in all_cols - set(s_pos.columns): s_pos[c] = pd.NA
    for c in all_cols - set(s_neg.columns): s_neg[c] = pd.NA
    table = pd.concat([s_pos[sorted(all_cols)], s_neg[sorted(all_cols)]], ignore_index=True)
    del s_pos, s_neg

    before = len(table)
    table = table.dropna(subset=[TIME]).copy()
    print(f"[Clean] dropna time: {before:,} → {len(table):,}")

    table['question_concept_id'] = pd.to_numeric(table['question_concept_id'], errors='coerce')
    n_bad_q = table['question_concept_id'].isna().sum()
    if n_bad_q:
        print(f"[WARN] question_concept_id NA {n_bad_q:,} rows，drop")
        table = table.dropna(subset=['question_concept_id']).copy()
    table['question_concept_id'] = table['question_concept_id'].astype('int64')
    return table

In [ ]:
def _drop_demo(table):
    q_text = table[['question_concept_id', 'question']].drop_duplicates()
    dup_qids = q_text.loc[q_text['question'].isin(DUP_QUESTIONS),
                          'question_concept_id'].unique().tolist()
    matched = set(q_text.loc[q_text['question_concept_id'].isin(dup_qids), 'question'])
    missing = set(DUP_QUESTIONS) - matched
    if missing:
        print(f"[WARN]  demo no matching:")
        for m in sorted(missing):
            print(f"        - {m}")

    n_before = len(table)
    table = table[~table['question_concept_id'].isin(dup_qids)].copy()
    print(f"[Step1] drop demo {len(dup_qids)}| rows {n_before:,} → {len(table):,}")
    return table

In [ ]:
PMI_DK  = 'PMI: Dont Know'
PMI_PNA = 'PMI: Prefer Not To Answer'


def _split_and_clean(table):
    is_skip  = table['answer'].astype(str).str.strip().eq('PMI: Skip')
    skipped  = table[is_skip].copy()
    answered = table[~is_skip].copy()
    print(f"[Split] real answer: {len(answered):,} | Skip: {len(skipped):,}")

    answered['answer_text'] = answered['answer'].astype(str).str.strip()
    answered['aid']    = pd.to_numeric(answered['answer_concept_id'], errors='coerce')
    answered['is_dk']  = answered['answer_text'].eq(PMI_DK)
    answered['is_pna'] = answered['answer_text'].eq(PMI_PNA)
    is_pmi = answered['is_dk'] | answered['is_pna']
    print(f" PMI answer: DK {answered['is_dk'].sum():,} | PNA {answered['is_pna'].sum():,}")

    qtype = (answered[~is_pmi].assign(miss=lambda d: d['aid'].isna())
             .groupby('question_concept_id')['miss'].mean()
             .rename('frac_missing_aid').reset_index())
    qtype['q_type'] = np.where(qtype['frac_missing_aid'] >= NUMERIC_QID_THR,
                               'numeric', 'categorical')
    all_qids = answered['question_concept_id'].drop_duplicates()
    qtype = (all_qids.to_frame().merge(qtype, on='question_concept_id', how='left'))
    qtype['q_type'] = qtype['q_type'].fillna('categorical')
    numeric_qids = set(qtype.loc[qtype['q_type'] == 'numeric', 'question_concept_id'])
    print(f" 题型: numeric {len(numeric_qids)} 道 | "
          f"categorical {(qtype['q_type'] == 'categorical').sum()} 道")

    is_num_q = answered['question_concept_id'].isin(numeric_qids)

    num_rows = answered[is_num_q].copy()
    raw = num_rows['answer_text']

    topcode = raw.str.extract(r'^(\d+)\s+or more', expand=False)
    n_topcode = topcode.notna().sum()
    if n_topcode:
        print(f" The answer shall be processed with the lower bound value: {n_topcode:,} ")
        print(raw[topcode.notna()].value_counts().head(5).to_string())

    num_rows['num_value'] = pd.to_numeric(raw.str.replace(',', '', regex=False), errors='coerce')
    num_rows.loc[topcode.notna(), 'num_value'] = pd.to_numeric(topcode[topcode.notna()])

    still_bad = num_rows['num_value'].isna() & ~num_rows['is_dk'] & ~num_rows['is_pna']
    n_invalid = still_bad.sum()
    print(f"numeric rows {len(num_rows):,} | illegal drop {n_invalid:,} "
          f"({n_invalid / max(len(num_rows), 1) * 100:.2f}%)")
    if n_invalid:
        print(f"[Step2] illgeal input:")
        print(num_rows.loc[still_bad, 'answer_text'].value_counts().head(10).to_string())
    num_rows = num_rows[~still_bad].copy()     

    cat_rows = answered[~is_num_q].copy()
    cat_pmi = cat_rows['is_dk'] | cat_rows['is_pna']
    n_bad_cat = (cat_rows['aid'].isna() & ~cat_pmi).sum()
    if n_bad_cat:
        bad_q = cat_rows.loc[cat_rows['aid'].isna() & ~cat_pmi, 'question'].value_counts().head(10)
        raise AssertionError(
            f"category none PMI {n_bad_cat:,} NA row answer_concept_id，different from expection:\n{bad_q.to_string()}")
    cat_rows.loc[~cat_pmi, 'aid'] = cat_rows.loc[~cat_pmi, 'aid'].astype('int64')
    print(f"category PMI {cat_pmi.sum():,} row")

    answered_clean = pd.concat([cat_rows, num_rows], ignore_index=True)
    print(f"cleaned answered: {len(answered):,} → {len(answered_clean):,}")
    return answered_clean, skipped, numeric_qids, qtype

In [ ]:
def _filter_response_rate(answered_clean, skipped, numeric_qids, qtype):
    participants = answered_clean['person_id'].nunique()
    print(f"paticipants: {participants:,}")

    q_response = (answered_clean.groupby(['question_concept_id', 'question'])['person_id']
                  .nunique().reset_index(name='n_answered'))
    q_response['response_rate'] = q_response['n_answered'] / participants
    q_response = q_response.merge(qtype[['question_concept_id', 'q_type']],
                                  on='question_concept_id', how='left')

    kept_qids = q_response.loc[q_response['response_rate'] >= RESPONSE_THR,
                               'question_concept_id'].tolist()
    print(f" question {len(q_response)} → keep {len(kept_qids)} "
          f"(answer rate ≥ {RESPONSE_THR * 100:.0f}%)")
    print(q_response[q_response['question_concept_id'].isin(kept_qids)]
          .sort_values('response_rate', ascending=False)
          [['question', 'q_type', 'response_rate']].to_string(index=False))

    answered_clean = answered_clean[answered_clean['question_concept_id'].isin(kept_qids)].copy()
    skipped        = skipped[skipped['question_concept_id'].isin(kept_qids)].copy()

    kept_numeric = [q for q in kept_qids if q in numeric_qids]
    kept_cat     = [q for q in kept_qids if q not in numeric_qids]
    return answered_clean, skipped, q_response, kept_numeric, kept_cat

In [ ]:
def _load_anchors(timeframe):
    a_pos = pd.read_csv(f'{CLEAN_DIR}/clean_positive_{timeframe}.csv');        a_pos['IsPositive'] = 1
    a_neg = pd.read_csv(f'{CLEAN_DIR}/clean_negative_anchor_{timeframe}.csv'); a_neg['IsPositive'] = 0
    anchors = pd.concat([a_pos, a_neg], ignore_index=True)[['person_id', 'IsPositive']]
    assert len(anchors) == N_TOTAL, f"Anchor error: {len(anchors)} != {N_TOTAL}"
    person_ids = anchors['person_id'].values
    pos_index  = pd.Series(np.arange(len(person_ids)), index=person_ids)
    print(f"[Anchor] {len(anchors):,} ✓")
    return anchors, person_ids, pos_index

In [ ]:
def _build_categorical(answered_clean, kept_cat, person_ids, pos_index):
    cat_part = answered_clean[answered_clean['question_concept_id'].isin(kept_cat)].copy()

    q_resp_persons = cat_part.groupby('question_concept_id')['person_id'].apply(set).to_dict()
    
    regular = cat_part[~(cat_part['is_dk'] | cat_part['is_pna'])].copy()
    regular['aid'] = regular['aid'].astype('int64')
    regular['qa_key'] = ('surv_q' + regular['question_concept_id'].astype(str)
                         + '__a' + regular['aid'].astype(str))

    n_opt = regular.groupby('question_concept_id')['aid'].nunique()
    print(f"{len(kept_cat)} categorical questions → {regular['qa_key'].nunique():,} regular binary columns "
          f"| options per question: median={n_opt.median():.0f}, max={n_opt.max()}")

    key_to_persons = regular.groupby('qa_key')['person_id'].apply(set).to_dict()
    key_to_qid     = regular.groupby('qa_key')['question_concept_id'].first().to_dict()

    cols = {}
    for key in sorted(key_to_persons):
        qid = key_to_qid[key]
        col = np.full(len(person_ids), np.nan, dtype='float32')
        idx = pos_index.reindex(list(q_resp_persons.get(qid, set()))).dropna().astype(int).values
        col[idx] = 0.0
        hidx = pos_index.reindex(list(key_to_persons[key])).dropna().astype(int).values
        col[hidx] = 1.0
        cols[key] = col

    flag_agg = (cat_part.groupby(['person_id', 'question_concept_id'])[['is_dk', 'is_pna']]
                .max().reset_index())
    n_flag_cols = 0
    for qid in kept_cat:
        base_idx = pos_index.reindex(list(q_resp_persons.get(qid, set()))).dropna().astype(int).values
        fsub = flag_agg[flag_agg['question_concept_id'] == qid]
        for flag, suffix in [('is_dk', 'dk'), ('is_pna', 'pna')]:
            hit = fsub.loc[fsub[flag], 'person_id'].values
            if len(hit) == 0:
                continue
            fcol = np.full(len(person_ids), np.nan, dtype='float32')
            fcol[base_idx] = 0.0
            hidx = pos_index.reindex(hit).dropna().astype(int).values
            fcol[hidx] = 1.0
            cols[f'surv_q{qid}__{suffix}'] = fcol
            n_flag_cols += 1
    print(f"categorical dk/pna columns: {n_flag_cols}")

    return cols, cat_part, regular

In [ ]:
def _build_numeric(answered_clean, kept_numeric, person_ids, pos_index):
    print(f"numerical {len(kept_numeric)}")
    num_part = answered_clean[answered_clean['question_concept_id'].isin(kept_numeric)]
    cols = {}
    if not len(num_part):
        return cols, num_part

    q_resp_persons = num_part.groupby('question_concept_id')['person_id'].apply(set).to_dict()
    num_agg = (num_part.dropna(subset=['num_value'])
               .groupby(['person_id', 'question_concept_id'])['num_value'].mean().reset_index())
    flag_agg = (num_part.groupby(['person_id', 'question_concept_id'])[['is_dk', 'is_pna']]
                .max().reset_index())

    for qid in kept_numeric:
        sub = num_agg[num_agg['question_concept_id'] == qid]
        col = np.full(len(person_ids), np.nan, dtype='float32')
        idx = pos_index.reindex(sub['person_id'].values)
        ok  = idx.notna().values
        col[idx[ok].astype(int).values] = sub['num_value'].values[ok]
        cols[f'surv_q{qid}__num'] = col

        base_idx = pos_index.reindex(list(q_resp_persons.get(qid, set()))).dropna().astype(int).values
        fsub = flag_agg[flag_agg['question_concept_id'] == qid]
        counts = {}
        for flag, suffix in [('is_dk', 'dk'), ('is_pna', 'pna')]:
            hit = fsub.loc[fsub[flag], 'person_id'].values
            counts[suffix] = len(hit)
            if len(hit) == 0:
                continue
            fcol = np.full(len(person_ids), np.nan, dtype='float32')
            fcol[base_idx] = 0.0
            hidx = pos_index.reindex(hit).dropna().astype(int).values
            fcol[hidx] = 1.0
            cols[f'surv_q{qid}__{suffix}'] = fcol

        print(f"    q{qid}: num n={ok.sum():,} median={np.nanmedian(col):.2f} "
              f"min={np.nanmin(col):.1f} max={np.nanmax(col):.1f} "
              f"| dk={counts['dk']:,} | pna={counts['pna']:,}")
    return cols, num_part

In [ ]:
def _assemble(anchors, new_cols, answered_clean, skipped):
    result = pd.concat([anchors[['person_id']].reset_index(drop=True),
                        pd.DataFrame(new_cols)], axis=1)

    total    = answered_clean.groupby('person_id').size().rename('surv_total_count').reset_index()
    unique_q = (answered_clean.groupby('person_id')['question_concept_id']
                .nunique().rename('surv_unique_question_count').reset_index())
    skip_cnt = skipped.groupby('person_id').size().rename('surv_skip_count').reset_index()
    for d in [total, unique_q, skip_cnt]:
        result = result.merge(d, on='person_id', how='left')
    for c in ['surv_total_count', 'surv_unique_question_count', 'surv_skip_count']:
        result[c] = result[c].fillna(0).astype('int32')

    print(f"Result shape: {result.shape}")

    chk = result.merge(anchors, on='person_id', how='left')
    pm  = chk.loc[chk.IsPositive == 1, 'surv_total_count'].mean()
    nm_ = chk.loc[chk.IsPositive == 0, 'surv_total_count'].mean()
    cov = (chk['surv_total_count'] > 0).sum()
    print(f"\n[Sanity] surv_total_count: pos={pm:.2f} | neg={nm_:.2f} | ratio={pm / max(nm_, 1e-6):.2f}x")
    print(f"[Sanity] covering: {cov:,}/{N_TOTAL} ({cov / N_TOTAL * 100:.1f}%)")
    sp = chk.loc[chk.IsPositive == 1, 'surv_skip_count'].mean()
    sn = chk.loc[chk.IsPositive == 0, 'surv_skip_count'].mean()
    print(f"[Sanity] surv_skip_count: pos={sp:.2f} | neg={sn:.2f}")

    feat_cols = [c for c in result.columns if c.startswith('surv_q')]
    na_rate = result[feat_cols].isna().mean()
    print(f"[Sanity] survey feature NA: median={na_rate.median() * 100:.1f}% "
          f"| min={na_rate.min() * 100:.1f}% | max={na_rate.max() * 100:.1f}%")
    return result, feat_cols

In [ ]:
def _build_name_map(cat_part, regular, num_part, q_response, kept_numeric, feat_cols):
    q_text = pd.concat([cat_part[['question_concept_id', 'question']],
                        num_part[['question_concept_id', 'question']]]).drop_duplicates()

    cat_map = (regular.groupby(['qa_key', 'question_concept_id', 'question',
                                'aid', 'answer_text'])['person_id']
               .nunique().reset_index(name='n_persons')
               .rename(columns={'aid': 'answer_concept_id', 'answer_text': 'answer'}))
    cat_map['feature_type'] = 'categorical'

    multi = (regular.drop_duplicates(['person_id', 'question_concept_id', 'aid'])
             .groupby(['question_concept_id', 'person_id']).size()
             .groupby('question_concept_id').max())
    cat_map['answer_type'] = cat_map['question_concept_id'].map(
        lambda q: 'multi_response' if multi.get(q, 1) > 1 else 'single_select')
    
    parts = []
    if len(kept_numeric):
        base = (num_part.dropna(subset=['num_value'])
                .groupby(['question_concept_id', 'question'])['person_id']
                .nunique().reset_index(name='n_persons'))
        base['qa_key']            = 'surv_q' + base['question_concept_id'].astype(str) + '__num'
        base['answer_concept_id'] = pd.NA
        base['answer']            = 'NUMERIC_VALUE'
        base['feature_type']      = 'numeric'
        base['answer_type']       = 'numeric'
        parts.append(base)

    all_flags = pd.concat([
        cat_part[['person_id', 'question_concept_id', 'is_dk', 'is_pna']],
        num_part[['person_id', 'question_concept_id', 'is_dk', 'is_pna']],
    ], ignore_index=True)
    for flag, suffix, label in [('is_dk', 'dk', PMI_DK), ('is_pna', 'pna', PMI_PNA)]:
        f = (all_flags[all_flags[flag]].groupby('question_concept_id')['person_id']
             .nunique().reset_index(name='n_persons'))
        if not len(f):
            continue
        f['qa_key'] = 'surv_q' + f['question_concept_id'].astype(str) + f'__{suffix}'
        f = f[f['qa_key'].isin(feat_cols)]
        f = f.merge(q_text, on='question_concept_id', how='left')
        f['answer_concept_id'] = pd.NA
        f['answer']            = label
        f['feature_type']      = 'pmi_flag'
        f['answer_type']       = 'pmi_flag'
        parts.append(f)

    name_map = pd.concat([cat_map] + parts, ignore_index=True)
    return (name_map.merge(q_response[['question_concept_id', 'response_rate']],
                           on='question_concept_id', how='left')
            .sort_values('n_persons', ascending=False))

In [ ]:
def build_survey_features(timeframe: int):
    print(f"SURVEY FEATURES — {timeframe} months")

    table = _load_survey(timeframe)
    table = _drop_demo(table)

    answered_clean, skipped, numeric_qids, qtype = _split_and_clean(table)
    answered_clean, skipped, q_response, kept_numeric, kept_cat = \
        _filter_response_rate(answered_clean, skipped, numeric_qids, qtype)

    anchors, person_ids, pos_index = _load_anchors(timeframe)

    cat_cols, cat_part, regular = _build_categorical(answered_clean, kept_cat, person_ids, pos_index)
    num_cols, num_part = _build_numeric(answered_clean, kept_numeric, person_ids, pos_index)
    new_cols = {**cat_cols, **num_cols}

    result, feat_cols = _assemble(anchors, new_cols, answered_clean, skipped)
    name_map = _build_name_map(cat_part, regular, num_part, q_response, kept_numeric, feat_cols)
    _check_and_save(result, name_map, feat_cols, timeframe)

    return result, name_map

In [ ]:
for tf in [6]:
    build_survey_features(tf)

In [ ]:
for tf in [12,24]:
    build_survey_features(tf)

In [ ]:
import pandas as pd

BASE        = '/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
MATRIX_DIR  = f'{BASE}/amia/matrix'          
NEWSURV_DIR = f'{BASE}/amia/new_survey'      
N_TOTAL = 267747

for tf in [6, 12, 24]:
    ehrdemo = pd.read_parquet(f'{MATRIX_DIR}/matrix_ehrdemo_{tf}.parquet')
    surv    = pd.read_parquet(f'{NEWSURV_DIR}/survey_features_{tf}.parquet')

    assert len(ehrdemo) == N_TOTAL, f"ehrdemo_{tf} row number {len(ehrdemo)} != {N_TOTAL}"
    assert len(surv)    == N_TOTAL, f"survey_{tf} row number {len(surv)} != {N_TOTAL}"

    ehrdemo['person_id'] = ehrdemo['person_id'].astype('int64')
    surv['person_id']    = surv['person_id'].astype('int64')

    overlap = (set(ehrdemo.columns) & set(surv.columns)) - {'person_id'}
    assert not overlap, f"[{tf}m] column conflict: {sorted(overlap)[:10]}"

    m = ehrdemo.merge(surv, on='person_id', how='left')
    assert len(m) == N_TOTAL, f"row error after merging: {len(m)}"

    surv_cols = [c for c in surv.columns if c != 'person_id']
    matched = m[surv_cols].notna().any(axis=1).sum()
    assert matched > 0, f"[{tf}m] merge is not in person_id"

    m.to_parquet(f'{NEWSURV_DIR}/matrix_ehrdemo_survnobasic_{tf}.parquet', index=False)

    n_ehr  = ehrdemo.shape[1] - 2     
    n_surv = len(surv_cols)
    print(f"[{tf}m] ehrdemo {n_ehr} + survey {n_surv} → {m.shape[1]-2} features "
          f"| shape {m.shape} | survey people with value {matched:,} ({matched/N_TOTAL*100:.1f}%) ✓")

print(f"\n[Save] → {NEWSURV_DIR}/matrix_ehrdemo_survnobasic_{{6,12,24}}.parquet")

In [ ]:
import pandas as pd
BASE = "/home/jupyter/workspace/rw-migration-aou-rw-24b38658/amia"

sv = pd.read_csv(BASE + "/surveys.csv")
print("shape:", sv.shape)
print("cols:", sv.columns.tolist())
print(sv.head(3))
print("\nonly person_id:", sv["person_id"].nunique())

CD = BASE + "/clean_data"
pos = set(pd.read_csv(f"{CD}/clean_positive_6.csv", usecols=["person_id"]).person_id)
neg = set(pd.read_csv(f"{CD}/clean_negative_anchor_6.csv", usecols=["person_id"]).person_id)
svp = set(sv["person_id"])
print(f"\npositive in surveys.csv: {len(pos & svp):,} / {len(pos):,} = {len(pos&svp)/len(pos):.1%}")
print(f"negative in surveys.csv: {len(neg & svp):,} / {len(neg):,} = {len(neg&svp)/len(neg):.1%}")

print("\ndate column:", [c for c in sv.columns if 'date' in c.lower() or 'time' in c.lower()])

In [ ]:
import pandas as pd, numpy as np, re

BASE = "/home/jupyter/workspace/rw-migration-aou-rw-24b38658/amia"
CD   = BASE + "/clean_data"
NEW  = BASE + "/new_survey"
SURVEYS  = BASE + "/surveys.csv"
POS_ANCH = CD + "/clean_positive_{tf}.csv"
NEG_ANCH = CD + "/clean_negative_anchor_{tf}.csv"
NB_TMPL  = NEW + "/survey_nobasic_{tf}.csv"

NONRESPONSE = {903096, 903079, 903087}   
TFS = [6, 12, 24]
USE_NOBASIC     = True
RESPONDERS_ONLY = True
INCLUDE_DAYS    = True
OUT = BASE + "/survey_availability_table.csv"

def med_iqr(x):
    x = pd.Series(x).dropna()
    if len(x)==0: return "—"
    q1,me,q3 = x.quantile([.25,.5,.75]); return f"{me:.0f} ({q1:.0f}–{q3:.0f})"

def to_date(s):
    return pd.to_datetime(s.astype(str).str.slice(0,10), format="%Y-%m-%d", errors="coerce")

def nobasic_qids(tf):
    head = pd.read_csv(NB_TMPL.format(tf=tf), nrows=0).columns.tolist()
    return {int(m.group(1)) for c in head for m in [re.match(r"surv_q(\d+)__", c)] if m}

SV_ALL = pd.read_csv(SURVEYS, usecols=["person_id","survey_datetime","question_concept_id","answer_concept_id"])
SV_ALL["survey_datetime"] = to_date(SV_ALL["survey_datetime"])
print(f"  {len(SV_ALL):,} 行, {SV_ALL.person_id.nunique():,} 人")

def load_cohort(tf):
    cols = ["person_id","index_date","timeframe_start"]
    p = pd.read_csv(POS_ANCH.format(tf=tf), usecols=cols); p["y"]=1
    n = pd.read_csv(NEG_ANCH.format(tf=tf), usecols=cols); n["y"]=0
    coh = pd.concat([p,n], ignore_index=True)
    coh["index_date"]      = to_date(coh["index_date"])
    coh["timeframe_start"] = to_date(coh["timeframe_start"])
    return coh

def per_tf(tf):
    coh = load_cohort(tf)
    qids = nobasic_qids(tf) if USE_NOBASIC else None

    sv = SV_ALL.merge(coh[["person_id","index_date","timeframe_start"]], on="person_id", how="inner")
    sv = sv[(sv["survey_datetime"] >= sv["timeframe_start"]) &
            (sv["survey_datetime"] <  sv["index_date"])]        
    if USE_NOBASIC:
        sv = sv[sv["question_concept_id"].isin(qids)]

    is_skip = sv["answer_concept_id"].isin(NONRESPONSE)         
    ans = sv.loc[~is_skip, ["person_id","question_concept_id"]].drop_duplicates()
    skp = sv.loc[ is_skip, ["person_id","question_concept_id"]].drop_duplicates()
    skp = skp.merge(ans, on=["person_id","question_concept_id"], how="left", indicator=True)
    skp = skp[skp["_merge"]=="left_only"]                       

    answered = ans.groupby("person_id")["question_concept_id"].nunique().rename("answered")
    skipped  = skp.groupby("person_id")["question_concept_id"].nunique().rename("skipped")
    latest   = sv.groupby("person_id")["survey_datetime"].max().rename("latest")
    nrows    = sv.groupby("person_id").size().rename("_nrows")

    st = coh.set_index("person_id").join([answered, skipped, latest, nrows])
    st["has_any"]       = st["_nrows"].fillna(0) > 0             
    st["answered"]      = st["answered"].fillna(0).astype(int)
    st["skipped"]       = st["skipped"].fillna(0).astype(int)
    st["days_to_index"] = (st["index_date"] - st["latest"]).dt.days
    return st

rows = []
for tf in TFS:
    st = per_tf(tf)
    print(f"\n=== tf={tf}: cohort={len(st):,} (pos={int((st.y==1).sum()):,}, neg={int((st.y==0).sum()):,}) ===")
    for lab, name in [(1,"OUD-positive"), (0,"OUD-negative")]:
        d = st[st.y==lab]
        base = d[d["has_any"]] if RESPONDERS_ONLY else d
        print(f"  {name}: interaction within the window={int(d['has_any'].sum()):,} ({d['has_any'].mean():.1%})")
        row = {
            "Look-back window": f"{tf} months", "Group": name,
            "Participants, n": f"{len(d):,}",
            "Participants with ≥1 eligible survey response, n (%)": f"{int(d['has_any'].sum()):,} ({100*d['has_any'].mean():.1f}%)",
            "Questions answered per respondent, median (IQR)": med_iqr(base["answered"]),
            "Skipped or declined questions, median (IQR)": med_iqr(base["skipped"]),
        }
        if INCLUDE_DAYS:
            row["Days from latest survey to index, median (IQR)"] = med_iqr(base["days_to_index"])
        rows.append(row)

table = pd.DataFrame(rows)
table.to_csv(OUT, index=False)
print("\nsaved →", OUT)
table

In [ ]:
st12 = per_tf(12)
d = st12[(st12.y==0) & (st12.has_any)]
print(d["answered"].describe(percentiles=[.25,.5,.75]))
print("Q1 =", d["answered"].quantile(.25))

In [ ]:
st12 = per_tf(12)
d = st12[(st12.y==0) & (st12.has_any)]

low = d[d["answered"] <= 3]
print("low answer rate:", len(low), f"= {len(low)/len(d):.1%}")

BASE = "/home/jupyter/workspace/rw-migration-aou-rw-24b38658/amia"
low_pids = set(low.index[:5]) 
coh = load_cohort(12).set_index("person_id")
sv5 = SV_ALL[SV_ALL.person_id.isin(low_pids)].merge(
    coh[["index_date","timeframe_start"]], left_on="person_id", right_index=True)
sv5 = sv5[(sv5.survey_datetime >= sv5.timeframe_start) & (sv5.survey_datetime < sv5.index_date)]
sv5 = sv5[sv5.question_concept_id.isin(nobasic_qids(12))]
for pid in low_pids:
    sub = sv5[sv5.person_id==pid]
    print(f"\npid={pid}: within the window {len(sub)} , # of questions={sub.question_concept_id.nunique()}")
    print("  window:", coh.loc[pid,"timeframe_start"], "→", coh.loc[pid,"index_date"])
    print("  date distribution:", sorted(sub.survey_datetime.dt.date.unique())[:5])